# Day 71 — Vector stores

A **vector store** keeps (text, embedding) pairs and finds the nearest by **cosine
similarity**. This 20-line version is the same idea as Chroma/FAISS — just smaller.
> ✅ Offline.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
import hashlib, math

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

class VectorStore:
    def __init__(self):
        self.items = []   # (id, text, embedding)
    def add(self, id, text):
        self.items.append((id, text, cheap_embed(text)))
    def search(self, query, k=2):
        qe = cheap_embed(query)
        # TODO: score each item by cosine(qe, emb), sort desc, return top-k (score, id, text)
        raise NotImplementedError

# s = VectorStore()
# for i, t in enumerate(["cats purr", "dogs bark", "python is a language"]): s.add(f"d{i}", t)
# print(s.search("programming language"))

## 🔒 Solution

In [ ]:
import hashlib, math

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

class VectorStore:
    def __init__(self):
        self.items = []
    def add(self, id, text):
        self.items.append((id, text, cheap_embed(text)))
    def search(self, query, k=2):
        qe = cheap_embed(query)
        scored = sorted(((cosine(qe, emb), id, text) for id, text, emb in self.items), reverse=True)
        return scored[:k]

s = VectorStore()
for i, t in enumerate(["cats purr softly", "dogs bark loudly", "python is a language"]):
    s.add(f"d{i}", t)
print(s.search("programming language"))